# TRIBE v2 — Brain or Bait?
Run Meta's brain encoding model on a video and visualize cortical activation.

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Install dependencies
!git clone https://github.com/facebookresearch/tribev2.git
!pip install -e tribev2 --quiet
!pip install huggingface_hub "numpy==2.2.6" --quiet

# Must restart runtime after installing numpy — this triggers it automatically
import os
os.kill(os.getpid(), 9)

In [ ]:
# 3. HuggingFace login (needed for LLaMA 3.2)
from huggingface_hub import login
login()  # paste your HF token when prompted

In [ ]:
# 4. Mount Google Drive and set video path
from google.colab import drive
drive.mount('/content/drive')

# !! Update this path to where stim.mp4 is in your Drive
VIDEO_PATH = '/content/drive/MyDrive/stim.mp4'
OUT_DIR = '/content'

In [ ]:
# 5. Load model
import sys
sys.path.insert(0, '/content/tribev2')

from tribev2 import TribeModel

model = TribeModel.from_pretrained('facebook/tribev2')
print('Model loaded.')

In [ ]:
# 6. Run inference
import numpy as np

events = model.get_events_dataframe(video_path=VIDEO_PATH)
print(f'Events: {len(events)} rows')

preds, segments = model.predict(events)
print(f'Predictions shape: {preds.shape}')  # (T, 20484)

np.save(f'{OUT_DIR}/predictions.npy', preds)
print('Saved predictions.npy')

In [ ]:
# 7. Visualize
import matplotlib.pyplot as plt

avg = preds.mean(axis=0)
lh = avg[:10242]   # left hemisphere
rh = avg[10242:]   # right hemisphere

print(f'Left  — mean: {lh.mean():.4f}, max: {lh.max():.4f}')
print(f'Right — mean: {rh.mean():.4f}, max: {rh.max():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(lh, bins=100, color='steelblue', edgecolor='none')
axes[0].set_title('Left Hemisphere Activation')
axes[0].set_xlabel('Predicted fMRI signal')
axes[1].hist(rh, bins=100, color='tomato', edgecolor='none')
axes[1].set_title('Right Hemisphere Activation')
axes[1].set_xlabel('Predicted fMRI signal')
fig.suptitle(f'TRIBE v2 — Brain Activation: stim.mp4\n({preds.shape[0]} time points)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/activation_histogram.png', dpi=150)
plt.show()
print('Done.')

In [ ]:
# 8. Download outputs
from google.colab import files
files.download(f'{OUT_DIR}/predictions.npy')
files.download(f'{OUT_DIR}/activation_histogram.png')